# 02. T2I·I2I API payload dry run

목표: 공식 Alibaba Cloud 메시지 형식에 맞춰 T2I와 I2I payload를 만들고 로컬에서 검증합니다. 네트워크 요청, API key 사용과 비용 발생은 없습니다.

2026-07-24 기준 모델은 초대 테스트 단계이며, 실제 호출에는 지역에 맞는 Workspace ID, endpoint와 API key가 필요합니다.

In [ ]:
import json
from urllib.parse import urlparse

MODEL_ID = "qwen-image-3.0-pro"

def text_to_image_payload(prompt, *, size="1536*1024", seed=42, prompt_extend=False):
    return {
        "model": MODEL_ID,
        "input": {
            "messages": [{
                "role": "user",
                "content": [{"text": prompt}],
            }]
        },
        "parameters": {
            "prompt_extend": prompt_extend,
            "n": 1,
            "size": size,
            "seed": seed,
            "watermark": True,
        },
    }

In [ ]:
t2i = text_to_image_payload(
    "한국어 교육 포스터. 제목은 정확히 '확산 모델 입문'. 세 단계 흐름도를 왼쪽에서 오른쪽으로 배치.",
    prompt_extend=False,
)
print(json.dumps(t2i, ensure_ascii=False, indent=2))

## I2I payload

참조 이미지는 배열 순서대로 역할을 적습니다. 예제 URL은 형식 검사만 위한 가상 주소이므로 호출하지 않습니다.

In [ ]:
def image_to_image_payload(image_urls, instruction, *, seed=42):
    content = [{"image": url} for url in image_urls]
    content.append({"text": instruction})
    return {
        "model": MODEL_ID,
        "input": {"messages": [{"role": "user", "content": content}]},
        "parameters": {
            "prompt_extend": False,
            "n": 1,
            "seed": seed,
            "watermark": True,
        },
    }

i2i = image_to_image_payload(
    ["https://example.com/person.png", "https://example.com/outfit.png"],
    "첫 번째 이미지의 얼굴과 자세를 보존하고 두 번째 이미지의 의상만 적용. 배경과 카메라 각도는 변경 금지.",
)
print(json.dumps(i2i, ensure_ascii=False, indent=2))

## Local validator

실제 API의 모든 검증 규칙을 재현하지는 않습니다. 자주 생기는 메시지 구조, 이미지 수, 출력 수와 해상도 오류를 요청 전에 찾는 학습용 검사기입니다.

In [ ]:
def parse_size(value):
    try:
        width, height = map(int, value.split("*"))
        return width, height
    except (AttributeError, ValueError):
        return None

def validate_payload(payload):
    errors = []
    if payload.get("model") != MODEL_ID:
        errors.append("model ID가 다릅니다.")
    messages = payload.get("input", {}).get("messages", [])
    if len(messages) != 1 or messages[0].get("role") != "user":
        errors.append("user message는 정확히 하나여야 합니다.")
        return errors
    content = messages[0].get("content", [])
    texts = [item.get("text") for item in content if "text" in item]
    images = [item.get("image") for item in content if "image" in item]
    if len(texts) != 1 or not texts[0].strip():
        errors.append("비어 있지 않은 text 항목이 하나 필요합니다.")
    if len(images) not in (0, 1, 2, 3):
        errors.append("참조 이미지는 0~3장이어야 합니다.")
    for image in images:
        parsed = urlparse(image)
        if parsed.scheme not in ("http", "https", "data"):
            errors.append(f"지원하지 않는 이미지 주소: {image}")
    parameters = payload.get("parameters", {})
    if not 1 <= parameters.get("n", 1) <= 6:
        errors.append("n은 1~6이어야 합니다.")
    if "size" in parameters:
        size = parse_size(parameters["size"])
        if not size or any(side < 512 or side > 2048 for side in size):
            errors.append("width와 height는 각각 512~2048 범위여야 합니다.")
    return errors

for name, payload in [("T2I", t2i), ("I2I", i2i)]:
    print(name, validate_payload(payload) or "검증 통과")

운영 코드에서는 key를 노트북에 적지 말고 `DASHSCOPE_API_KEY` 환경 변수나 비밀 관리 시스템에서 읽어야 합니다. 또한 지역별 key와 endpoint를 섞지 말고, 반환 이미지 URL이 만료되기 전에 결과를 저장해야 합니다.